In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import joblib
import pandas as pd

reg_model = joblib.load('/content/drive/MyDrive/EnviroSense/models/reg_fp32.pkl')
clf_model = joblib.load('/content/drive/MyDrive/EnviroSense/models/clf_fp32.pkl')

test_df = pd.read_csv('/content/drive/MyDrive/EnviroSense/data/test.csv')

X_test = test_df[['temperature', 'humidity', 'gas']]
y_test_reg = test_df['risk_index']
y_test_clf = test_df['anomaly_label']

print("Models and test data loaded")
print(reg_model)
print(clf_model)

Models and test data loaded
MLPRegressor(hidden_layer_sizes=(8,), max_iter=1000, random_state=42)
MLPClassifier(hidden_layer_sizes=(8,), max_iter=1000, random_state=42)


In [3]:
# Look at the actual learned weights inside the regression model
print("Number of weight matrices:", len(reg_model.coefs_))
print("Shape of each weight matrix:")
for i, w in enumerate(reg_model.coefs_):
    print(f"  Layer {i}: {w.shape}")

print("\nFirst few weight values (layer 0):")
print(reg_model.coefs_[0][:3])

print("\nData type of weights:", reg_model.coefs_[0].dtype)

Number of weight matrices: 2
Shape of each weight matrix:
  Layer 0: (3, 8)
  Layer 1: (8, 1)

First few weight values (layer 0):
[[ 3.01152765e-36  7.74188093e-01  8.33803904e-02  3.78313875e-02
  -5.97840800e-01 -1.15550868e-10 -2.86674277e-02  4.18928895e-01]
 [-3.47938243e-44  1.27996488e-01 -8.05268026e-01  5.84655810e-01
   6.63207735e-01 -2.51213009e-13 -2.65504747e-02 -7.74628847e-01]
 [-4.41875740e-21  1.48158699e-01  4.70788117e-02 -3.93451615e-01
   7.31919989e-02 -4.76786762e-10 -2.70120661e-02  8.97242573e-02]]

Data type of weights: float64


In [5]:
import numpy as np

def quantize_weights(weights):
    """Converting FP32 weights to INT8, returning the quantized weights
    plus the scale factor needed to convert back."""

    # Finding the actual range of values in this weight table
    w_min = weights.min()
    w_max = weights.max()

    # Calculating the scale: how much one INT8 "step" represents in real value
    scale = (w_max - w_min) / 255.0

    # Shifting and scaling the weights into the 0-255 range, then re-center to -128..127
    quantized = np.round((weights - w_min) / scale - 128)
    quantized = np.clip(quantized, -128, 127).astype(np.int8)

    return quantized, scale, w_min

# Testing it on layer 0 of the regression model
q_weights, scale, w_min = quantize_weights(reg_model.coefs_[0])

print("Original weights (first row):", reg_model.coefs_[0][0])
print("Quantized weights (first row):", q_weights[0])
print("Scale factor:", scale)
print("Data type now:", q_weights.dtype)

Original weights (first row): [ 3.01152765e-36  7.74188093e-01  8.33803904e-02  3.78313875e-02
 -5.97840800e-01 -1.15550868e-10 -2.86674277e-02  4.18928895e-01]
Quantized weights (first row): [  2 127  15   8 -95   2  -3  70]
Scale factor: 0.006193945562500222
Data type now: int8


In [6]:
def dequantize_weights(quantized, scale, w_min):
    """Converting INT8 weights back to approximate decimal values."""
    return (quantized.astype(np.float64) + 128) * scale + w_min

def quantize_model(model):
    """Quantizing every weight layer in a model, then rebuilding a
    dequantized version"""

    quantized_layers = []
    dequantized_layers = []

    for layer_weights in model.coefs_:
        q, scale, w_min = quantize_weights(layer_weights)
        quantized_layers.append(q)

        dq = dequantize_weights(q, scale, w_min)
        dequantized_layers.append(dq)

    return quantized_layers, dequantized_layers

# Quantizing the regression model
reg_quantized, reg_dequantized = quantize_model(reg_model)

# Quantizing the classification model
clf_quantized, clf_dequantized = quantize_model(clf_model)

print("Regression model layers quantized:", len(reg_quantized))
print("Classification model layers quantized:", len(clf_quantized))

Regression model layers quantized: 2
Classification model layers quantized: 2


In [10]:
# Recalculating FP32 baseline metrics in this notebook
y_pred_reg = reg_model.predict(X_test)
mse = mean_squared_error(y_test_reg, y_pred_reg)

y_pred_clf = clf_model.predict(X_test)
acc = accuracy_score(y_test_clf, y_pred_clf)
prec = precision_score(y_test_clf, y_pred_clf)
rec = recall_score(y_test_clf, y_pred_clf)

print("FP32 MSE:", mse)
print("FP32 Accuracy:", acc, "Precision:", prec, "Recall:", rec)

FP32 MSE: 8.986709457218778
FP32 Accuracy: 0.995 Precision: 0.975609756097561 Recall: 1.0


In [12]:
import copy
from sklearn.metrics import mean_squared_error, accuracy_score, precision_score, recall_score

def make_quantized_model(original_model, dequantized_weights):
    """Create a copy of the model with quantized weights swapped in."""
    quantized_model = copy.deepcopy(original_model)
    quantized_model.coefs_ = dequantized_weights
    return quantized_model

# Building quantized versions of both models
reg_model_q = make_quantized_model(reg_model, reg_dequantized)
clf_model_q = make_quantized_model(clf_model, clf_dequantized)

# Running predictions with the quantized regression model
y_pred_reg_q = reg_model_q.predict(X_test)
mse_q = mean_squared_error(y_test_reg, y_pred_reg_q)

# Running predictions with the quantized classification model
y_pred_clf_q = clf_model_q.predict(X_test)
acc_q = accuracy_score(y_test_clf, y_pred_clf_q)
prec_q = precision_score(y_test_clf, y_pred_clf_q)
rec_q = recall_score(y_test_clf, y_pred_clf_q)

print("Regression: FP32 vs INT8")
print("FP32 MSE:", mse)
print("INT8 MSE:", mse_q)

print("\nClassification: FP32 vs INT8")
print("FP32 Accuracy:", acc, "| INT8 Accuracy:", acc_q)
print("FP32 Precision:", prec, "| INT8 Precision:", prec_q)
print("FP32 Recall:", rec, "| INT8 Recall:", rec_q)

Regression: FP32 vs INT8
FP32 MSE: 8.986709457218778
INT8 MSE: 9.248774274313966

Classification: FP32 vs INT8
FP32 Accuracy: 0.995 | INT8 Accuracy: 0.995
FP32 Precision: 0.975609756097561 | INT8 Precision: 0.975609756097561
FP32 Recall: 1.0 | INT8 Recall: 1.0


In [14]:
# Calculating size in bytes for FP32 weights (original)
reg_fp32_size = sum(w.nbytes for w in reg_model.coefs_)
clf_fp32_size = sum(w.nbytes for w in clf_model.coefs_)

# Calculating size in bytes for INT8 weights (quantized)
reg_int8_size = sum(w.nbytes for w in reg_quantized)
clf_int8_size = sum(w.nbytes for w in clf_quantized)

print("Regression Model Size")
print("FP32 size:", reg_fp32_size, "bytes")
print("INT8 size:", reg_int8_size, "bytes")
print("Compression ratio:", round(reg_fp32_size / reg_int8_size, 2), "x smaller")

print("\nClassification Model Size")
print("FP32 size:", clf_fp32_size, "bytes")
print("INT8 size:", clf_int8_size, "bytes")
print("Compression ratio:", round(clf_fp32_size / clf_int8_size, 2), "x smaller")

Regression Model Size
FP32 size: 256 bytes
INT8 size: 32 bytes
Compression ratio: 8.0 x smaller

Classification Model Size
FP32 size: 256 bytes
INT8 size: 32 bytes
Compression ratio: 8.0 x smaller


In [15]:
import joblib

# Save the raw INT8 weight arrays (the actual compressed format)
joblib.dump(reg_quantized, '/content/drive/MyDrive/EnviroSense/models/reg_int8_weights.pkl')
joblib.dump(clf_quantized, '/content/drive/MyDrive/EnviroSense/models/clf_int8_weights.pkl')

print("INT8 weight arrays saved")

INT8 weight arrays saved


## Day 2 - FP32 vs INT8 Quantization Results

### Regression Model (risk_index prediction)

| Metric          | FP32      | INT8      | Change        |
|-----------------|-----------|-----------|---------------|
| MSE             | 8.987     | 9.249     | +0.26 (~3%)   |
| Model size      | 256 bytes | 32 bytes  | 8x smaller    |

### Classification Model (NORMAL vs ANOMALY)

| Metric          | FP32      | INT8      | Change        |
|-----------------|-----------|-----------|---------------|
| Accuracy        | 99.5%     | 99.5%     | No change     |
| Precision       | 97.6%     | 97.6%     | No change     |
| Recall          | 100%      | 100%      | No change     |
| Model size      | 256 bytes | 32 bytes  | 8x smaller    |

### Method
Weights were manually quantized from FP32 (64-bit float) to INT8
(8-bit integer) using min-max scaling: each weight table's range was
mapped onto the -128 to 127 INT8 range using a per-layer scale factor,
then dequantized back to approximate decimals to test real prediction
accuracy (simulating on-device inference behavior).

### Conclusion
Quantization achieved an exact 8x model compression (consistent with
the 8-byte-to-1-byte reduction per weight) with zero accuracy loss on
classification and a negligible ~3% increase in regression error. This
confirms the models are well-suited for edge deployment on memory-
constrained hardware like the ESP32, without a meaningful accuracy
tradeoff.

**Files saved:** `models/reg_int8_weights.pkl`, `models/clf_int8_weights.pkl`